# Asystent AI

## Plan

Na google drive wrzuciłem 3 artykuły na temat fizyki:
- Absorption spectrum of very low pressure,
relatively cold atomic hydrogen.
- Anti-photon.
- Comparison of Monte-Carlo and Einstein methods in the light-gas interactions.

W ich doborze skupiałem się głównie na tym, żeby nie było zbyt dużo wzorów.

W rozwiązaniu użyje modelu embeddingu: `sentence-transformers/all-MiniLM-L6-v2` i dwóch różnych modeli językowych, które porównam: `google/flan-t5-base` i `declare-lab/flan-alpaca-base`.

Do stworzenia bazy wektorów użyłem FAISS, zpromptowałem LLM-y, aby uniknąć halucynacji i stworzyłem mały zestaw testowy.

Poprawne odpowiedzi na pytania testowe to:

- No.
- I don't know.
- The Supernova SNR1987A is an example of the utility of optical coherence...
- I don't know
- I don't know

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install langchain sentence-transformers faiss-cpu pypdf transformers torch langchain-community  -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [15]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline, AutoModelForSeq2SeqLM
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
import os
from transformers import pipeline
import torch

In [5]:
folder_path = '/content/drive/My Drive/papiery/'

docs = []
for file in os.listdir(folder_path):
    if file.endswith('.pdf'):
        path = os.path.join(folder_path, file)
        loader = PyPDFLoader(path)
        pages = loader.load()

        for page in pages:
            page.metadata['source_file'] = file
        docs.extend(pages)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(docs)

In [6]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

vector_db = FAISS.from_documents(chunks, embeddings)
vector_db.save_local("faiss_index_hf")

/tmp/ipython-input-559570358.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
def get_chain(llm_model_name, memory=None):
    chat_prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            "You are an assistant that always relies only on the provided context. "
            "Never make up information. "
            "If the context does not contain the answer, respond exactly: "
            "\"I don't know.\""
            "Do not add extra commentary, guesses, or unrelated information."
            ),
        HumanMessagePromptTemplate.from_template(
            "Context: {context}\n\nQuestion: {question}\n\n"
            )
    ])

    tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

    pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens = 256,
        repetition_penalty=1.5
    )

    llm = HuggingFacePipeline(pipeline=pipe)

    if memory is not None:
        chain = ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=vector_db.as_retriever(search_kwargs={'k': 5}),
            memory=memory,
            combine_docs_chain_kwargs={"prompt": chat_prompt}
        )
    else:
        chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vector_db.as_retriever(search_kwargs={'k': 5}),
            chain_type_kwargs={"prompt": chat_prompt},
            return_source_documents=True,
        )


    return chain

chain1 = get_chain("google/flan-t5-base")
chain2 = get_chain("declare-lab/flan-alpaca-base")

Device set to use cpu
Device set to use cpu


In [26]:
def ask(chain, question):
    # Adjust input key based on whether the chain has memory
    if hasattr(chain, 'memory') and chain.memory is not None:
        inputs = {'question': question}
    else:
        inputs = {'query': question}

    result = chain.invoke(inputs)
    if 'result' in result:
        answer = result['result']
    elif 'answer' in result:
        answer = result['answer']
    else:
        answer = "Could not retrieve an answer."


    sources = []
    if 'source_documents' in result:
         for doc in result['source_documents']:
            source_info = {
                'source_file' : doc.metadata.get('source_file', 'unknown'),
                'page': doc.metadata.get('page', -1) + 1
            }
            sources.append(source_info)


    print(f'Ai Assistant: {answer}')
    for i, source in enumerate(sources, start=1):
        print(f'{i}. Source File: {source["source_file"]}, Page: {source["page"]}')

In [20]:
def test(chain):
    ask(chain, 'Does the atomic theory shows that the sizes of the electric charges are much bigger tha their distances?')
    ask(chain, 'What does Planck\'s formule reveal?')
    ask(chain, 'What is the Supernova SNR1987A example of?')
    ask(chain, 'What is the current president of USA?')
    ask(chain, 'What day was 18-th November 1999?')

In [21]:
test(chain1)

Token indices sequence length is longer than the specified maximum sequence length for this model (1159 > 512). Running this sequence through the model will result in indexing errors


Ai Assistant: No, the charge is much larger close to it than the field emitted by other charg es in similar conditions of radiation: Thus, the charge is much larger close to it than the field emitted by other charg es in similar conditions of radiation: Thus, a large number of other charges is needed, it seem s that it remains a “residual field”.
1. Source File: Anti-photon.pdf, Page: 2
2. Source File: Anti-photon.pdf, Page: 2
3. Source File: Anti-photon.pdf, Page: 10
4. Source File: Anti-photon.pdf, Page: 3
5. Source File: Anti-photon.pdf, Page: 2
Ai Assistant: The choice of normal modes in acoustics is not arbitrary beca use the equations of propagation of sound are not strictly linear: the “normal modes” correspond to the be st linear approximation of the equations. Subsection 2.1 shows that the linearity of Maxwell’s equations in the va cuum may be extended in matter. Subsection 2.2 uses this linearity to define precisely the vector spac e of the solutions of Maxwell’s equations, t

In [12]:
test(chain2)

Token indices sequence length is longer than the specified maximum sequence length for this model (1159 > 512). Running this sequence through the model will result in indexing errors


Ai Assistant: No, the atomic theory does not show that the sizes of the electric charges are much bigger than their distances.
1. Source File: Anti-photon.pdf, Page: 2
2. Source File: Anti-photon.pdf, Page: 2
3. Source File: Anti-photon.pdf, Page: 10
4. Source File: Anti-photon.pdf, Page: 3
5. Source File: Anti-photon.pdf, Page: 2
Ai Assistant: I don't know.
1. Source File: Anti-photon.pdf, Page: 3
2. Source File: Anti-photon.pdf, Page: 2
3. Source File: Anti-photon.pdf, Page: 2
4. Source File: Monte-carlo-Einstein.pdf, Page: 2
5. Source File: Monte-carlo-Einstein.pdf, Page: 1
Ai Assistant: The Supernova SNR1987A is an example of the utility of optical coherence in the interpretation of the aspect and the spectra of many punctuated rings, surrounding a central star possibly masked, observed in the sky. But the most important consequence is the ne gligence of a parameter interaction between luminous rays, catalyzed by excited atomic hyd rogen; this interaction often leads to a redshift 

## Pamietanie konwersacji

Aby model spamiętywał poprzednią konwersacje należy skorzystać z `ConversationBufferMemory` i `ConversationRetrievalChain`, który obsługuje pamięć i sam skleja historię rozmowy z kontekstem.

In [27]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

chain3 = get_chain("google/flan-t5-base", memory)

test(chain3)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1159 > 512). Running this sequence through the model will result in indexing errors


Ai Assistant: No, the charge is much larger close to it than the field emitted by other charg es in similar conditions of radiation: Thus, the charge is much larger close to it than the field emitted by other charg es in similar conditions of radiation: Thus, a large number of other charges is needed, it seem s that it remains a “residual field”.
Ai Assistant: In response to numerous criticisms, such as non equivalence of ene rgy of a mode to kTP for a large temperature TP , Planck amended his law in 1911 [4, 5], obtaining the absolute spectral 1The choice of normal modes in acoustics is not arbitrary beca use the equations of propagation of sound are not strictly linear: the “normal modes” correspond to the be st linear approximation of the equations. 3 the scale of their distances. Planck’s law corrected by its author in 1911 [4] and approved by Einst ein and Stern [5]. 2.1 Linearity of Maxwell’s equations in matter. The equations of the electromagnetic field in vacuum, united under 

## Wnioski

Modele radzą sobie całkiem dobrze. Były w stanie wydobyć istotne fragmenty z bazy, ponieważ zarówno ich odpowiedzi, jak i wskazane strony dokumentów pokrywały się z prawdą. Halucynacje zaobserwowałem jedynie przy drugim pytaniu. Mogło się tak stać, ponieważ w artykule pojawia się jedynie krótka wzmianka o wzorze Plancka.